# Práctica 3 Representaciones Vectoriales

## Reutilizamos las funciones vistas en el notebook de clase.

In [1]:
import re
import pandas as pd 
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/luis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
def simple_preprocess(text: str):
    tokens = word_tokenize(text.lower(), language='spanish')
    return [word for word in tokens if word.isalnum() and len(word) > 1 and not re.match(r'^\d+$', word)]

In [3]:
def create_bow_dataframe(docs_raw: list, titles: list[str], vectorizer) -> pd.DataFrame:
    # fit_transform ajusta el vocabulario y crea la matriz en un solo paso
    matrix = vectorizer.fit_transform(docs_raw)

    # Podemos crear el DataFrame directamente pasando la matriz a un array tradicional
    # vectorizer.get_feature_names_out() nos da la lista de palabras en el orden exacto de las columnas
    df = pd.DataFrame(
        matrix.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
    )
    return df

In [4]:
doc1 = """Resolver el ajedrez por computadora, sin embargo, plantea un problema que no podrá
solucionarse incluso en el futuro: el crecimiento exponencial de las posibles jugadas y
variantes que pueden producirse."""
doc2 = """Deep Thought es el antecesor de la famosa máquina de ajedrez de IBM, Deep Blue 202.
Todo inició como un proyecto en la Universidad Carnegie Mellon, cuando un estudiante
graduado, Feng-hsiung Hsu, comenzó a trabajar en su proyecto de tesis: una máquina que
juega al ajedrez a la que él llamó ChipTest. """
doc3 = """ La verdad es que la teoría hipermoderna no es otra cosa que la aplicación, durante el desarrollo
de la apertura, de los mismos viejos principios, pero poniendo en práctica tácticas un tanto
novedosas."""
doc4 = """ Pensaba en ti, Susana. En las lomas verdes. Cuando volábamos papalotes[57] en la época del aire. Oíamos allá abajo el rumor viviente del pueblo mientras estábamos encima de él, arriba de la loma, en tanto se nos iba el hilo de cáñamo arrastrado por el viento. “Ayúdame, Susana.” Y unas manos suaves se apretaban a nuestras manos. “Suelta más hilo.”
»El aire nos hacía reír; juntaba la mirada de nuestros ojos, mientras el hilo corría entre los dedos detrás del viento, hasta que se rompía con un leve crujido como si hubiera sido trozado por las alas de algún pájaro. Y allá arriba, el pájaro de papel caía en maromas[58] arrastrando su cola de hilacho, perdiéndose en el verdor de la tierra.
»Tus labios estaban mojados como si los hubiera besado el rocío.»
"""
doc5 = """¿Por qué aquella mirada se volvía valiente ante la resignación? Qué le costaba a él perdonar, cuando era tan fácil decir una palabra o dos, o cien palabras si éstas fueran necesarias para salvar el alma. ¿Qué sabía él del Cielo y del Infierno?
Y sin embargo, él, perdido en un pueblo sin nombre, sabía los que habían merecido el Cielo. Había un catálogo. Comenzó a recorrer los santos del panteón católico comenzando por los del día: «Santa Nunilona, virgen y mártir; Anercio, obispo; Santas Salomé viuda, Alodia o Elodia y Nulina, vírgenes; Córdula y Donato.» Y siguió. Ya iba siendo dominado por el sueño cuando se sentó en la cama: «Estoy repasando una hilera de santos como si estuviera viendo saltar cabras.»
Salió fuera y miró el cielo. Llovían estrellas. Lamentó aquello porque hubiera querido ver un cielo quieto. Oyó el canto de los gallos. Sintió la envoltura de la noche cubriendo la tierra. La tierra, «este valle de lágrimas»."""
# El ratio dorado: 5 repeticiones para engañar a BoW y 16 rarezas para hackear TF-IDF
doc6 = "ajedrez ajedrez ajedrez ajedrez ajedrez lomas verdes papalotes viviente cáñamo suaves apretaban crujido trozado maromas hilacho rocío mojados besado verdor labios"

In [5]:
docs_raw = [doc1, doc2, doc3, doc4, doc5, doc6]

In [6]:
vectorizer = CountVectorizer(tokenizer=simple_preprocess,token_pattern=None)
bag_of_words_corpus = vectorizer.fit_transform(docs_raw)
diccionario = vectorizer.vocabulary_
sorted(diccionario.items(), key=lambda item: item[1])

[('abajo', 0),
 ('aire', 1),
 ('ajedrez', 2),
 ('al', 3),
 ('alas', 4),
 ('algún', 5),
 ('allá', 6),
 ('alma', 7),
 ('alodia', 8),
 ('anercio', 9),
 ('ante', 10),
 ('antecesor', 11),
 ('apertura', 12),
 ('aplicación', 13),
 ('apretaban', 14),
 ('aquella', 15),
 ('aquello', 16),
 ('arrastrado', 17),
 ('arrastrando', 18),
 ('arriba', 19),
 ('ayúdame', 20),
 ('besado', 21),
 ('blue', 22),
 ('cama', 23),
 ('canto', 24),
 ('carnegie', 25),
 ('catálogo', 26),
 ('católico', 27),
 ('caía', 28),
 ('chiptest', 29),
 ('cielo', 30),
 ('cien', 31),
 ('cola', 32),
 ('comenzando', 33),
 ('comenzó', 34),
 ('como', 35),
 ('computadora', 36),
 ('con', 37),
 ('corría', 38),
 ('cosa', 39),
 ('costaba', 40),
 ('crecimiento', 41),
 ('crujido', 42),
 ('cuando', 43),
 ('cubriendo', 44),
 ('cáñamo', 45),
 ('córdula', 46),
 ('de', 47),
 ('decir', 48),
 ('dedos', 49),
 ('deep', 50),
 ('del', 51),
 ('desarrollo', 52),
 ('detrás', 53),
 ('dominado', 54),
 ('dos', 55),
 ('durante', 56),
 ('día', 57),
 ('el', 58),
 

In [7]:
for column_idx, word in enumerate(vectorizer.get_feature_names_out()):
    print(column_idx,word)

0 abajo
1 aire
2 ajedrez
3 al
4 alas
5 algún
6 allá
7 alma
8 alodia
9 anercio
10 ante
11 antecesor
12 apertura
13 aplicación
14 apretaban
15 aquella
16 aquello
17 arrastrado
18 arrastrando
19 arriba
20 ayúdame
21 besado
22 blue
23 cama
24 canto
25 carnegie
26 catálogo
27 católico
28 caía
29 chiptest
30 cielo
31 cien
32 cola
33 comenzando
34 comenzó
35 como
36 computadora
37 con
38 corría
39 cosa
40 costaba
41 crecimiento
42 crujido
43 cuando
44 cubriendo
45 cáñamo
46 córdula
47 de
48 decir
49 dedos
50 deep
51 del
52 desarrollo
53 detrás
54 dominado
55 dos
56 durante
57 día
58 el
59 elodia
60 embargo
61 en
62 encima
63 entre
64 envoltura
65 era
66 es
67 estaban
68 este
69 estoy
70 estrellas
71 estudiante
72 estuviera
73 estábamos
74 exponencial
75 famosa
76 fuera
77 fueran
78 futuro
79 fácil
80 gallos
81 graduado
82 había
83 habían
84 hacía
85 hasta
86 hilacho
87 hilera
88 hilo
89 hipermoderna
90 hsu
91 hubiera
92 iba
93 ibm
94 incluso
95 infierno
96 inició
97 juega
98 jugadas
99 juntab

In [8]:
bag_of_words_corpus.toarray()

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 2, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 2, 0, ..., 1, 1, 0],
       [0, 0, 0, ..., 3, 0, 1],
       [0, 0, 5, ..., 0, 0, 0]], shape=(6, 234))

In [9]:
print(len(bag_of_words_corpus.toarray()))
len(bag_of_words_corpus.toarray()[1])

6


234

In [10]:
titulos = ["ajedrez por compu","DeepBlue","Fundamentos","Pedro Páramo1","Pedro Páramo2","query_tramposo"]
docs_matrix = create_bow_dataframe(
    docs_raw,
    titulos,
    vectorizer = CountVectorizer(tokenizer = simple_preprocess, token_pattern = None)
)

In [11]:
docs_matrix

,abajo,aire,ajedrez,al,alas,algún,allá,alma,alodia,anercio,...,virgen,viuda,viviente,volvía,volábamos,vírgenes,ya,él,época,éstas
ajedrez por compu,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DeepBlue,0,0,2,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
Fundamentos,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pedro Páramo1,1,2,0,0,1,1,2,0,0,0,...,0,0,1,0,1,0,0,1,1,0
Pedro Páramo2,0,0,0,0,0,0,0,1,1,1,...,1,1,0,1,0,1,1,3,0,1
query_tramposo,0,0,5,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [12]:
# 1. Extraemos el vector de la query y le damos la forma (1, -1) que pide sklearn
vector_query = docs_matrix.loc["query_tramposo"].values.reshape(1, -1)

# 2. Creamos una lista vacía para guardar los resultados y armar la tabla después
resultados_bow = []

# 3. Hacemos el for loop iterando sobre los 5 primeros títulos (sin incluir la query)
for titulo in titulos[:5]: 
    # Extraemos el vector del documento actual
    vector_doc = docs_matrix.loc[titulo].values.reshape(1, -1)
    
    # Calculamos la similitud (nos devuelve una matriz de 1x1, por eso el [0][0])
    similitud = cosine_similarity(vector_query, vector_doc)[0][0]
    
    # Guardamos el resultado
    resultados_bow.append((titulo, similitud))
    
    # Lo imprimimos para ir viéndolo
    print(f"Similitud Query vs {titulo}: {similitud:.4f}")

Similitud Query vs ajedrez por compu: 0.1284
Similitud Query vs DeepBlue: 0.1746
Similitud Query vs Fundamentos: 0.0000
Similitud Query vs Pedro Páramo1: 0.1388
Similitud Query vs Pedro Páramo2: 0.0000


### Ahora en TF-IDF

In [13]:
doc_matrix_tfidf = create_bow_dataframe(
    docs_raw,
    titulos,
    vectorizer = TfidfVectorizer(tokenizer = simple_preprocess, token_pattern = None)
)

In [14]:
vector_query_tfidf = doc_matrix_tfidf.loc["query_tramposo"].values.reshape(1,-1)
resultados_tfidf=[]
for titulo in titulos[:5]:
    vector_doc_tfidf = doc_matrix_tfidf.loc[titulo].values.reshape(1,-1)
    similitud_tfidf = cosine_similarity(vector_query_tfidf, vector_doc_tfidf)[0][0]
    resultados_tfidf.append((titulo,similitud_tfidf))
    print(f"similitud TD-IDF query vs {titulo}:{similitud_tfidf:.4f}")

similitud TD-IDF query vs ajedrez por compu:0.1051
similitud TD-IDF query vs DeepBlue:0.1453
similitud TD-IDF query vs Fundamentos:0.0000
similitud TD-IDF query vs Pedro Páramo1:0.1746
similitud TD-IDF query vs Pedro Páramo2:0.0000


In [15]:
resultados_tfidf

[('ajedrez por compu', np.float64(0.10513422171573941)),
 ('DeepBlue', np.float64(0.14533611774628666)),
 ('Fundamentos', np.float64(0.0)),
 ('Pedro Páramo1', np.float64(0.17464474059384852)),
 ('Pedro Páramo2', np.float64(0.0))]

In [16]:
scores_bow = [resultado[1] for resultado in resultados_bow]
scores_tfidf = [resultado[1] for resultado in resultados_tfidf]
nombre_docs = titulos[:5]
df_comp = pd.DataFrame({
    "Score BOW": scores_bow,
    "Score TF-IDF": scores_tfidf
}, index = nombre_docs)

In [17]:
df_comp

,Score BOW,Score TF-IDF
ajedrez por compu,0.128374,0.105134
DeepBlue,0.174608,0.145336
Fundamentos,0.000000,0.000000
Pedro Páramo1,0.138821,0.174645
Pedro Páramo2,0.000000,0.000000


#### Hubo un cambio del Score BoW a TF-IDF
Podemos notar que con la query tramposa en BoW la frecuencia de ‘ajedrez’ domina por lo que en el documento 3 Deep Blue comparte mayor similitud que en las demás, pero al tener más palabras únicas en comun con Pedro Páramo1, al hacer TF-IDF, dicho documento tiene muchas mayores coicidencias con la query tramposa.

## Parte 2: Busqueda de sesgos.

In [18]:
import gensim.downloader as gensim_api
word_vectors = gensim_api.load("glove-wiki-gigaword-100")

In [19]:
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))

[('practice', 0.6156836152076721), ('knowledge', 0.6129590272903442), ('teaching', 0.5949095487594604), ('skill', 0.5886170864105225), ('reputation', 0.588079571723938), ('philosophy', 0.5868663191795349), ('work', 0.5848589539527893), ('skills', 0.5772278904914856), ('discipline', 0.576593816280365), ('mind', 0.5739315152168274)]

[('professions', 0.6473134756088257), ('practitioner', 0.5966603755950928), ('nursing', 0.5942842364311218), ('vocation', 0.5698666572570801), ('teaching', 0.5623518824577332), ('childbirth', 0.543552815914154), ('academic', 0.5408717393875122), ('teacher', 0.5401058197021484), ('educator', 0.5207306742668152), ('qualifications', 0.5143449902534485)]


### Analizando los sesgos genéricos relacionados a las profesiones de hombre y mujeres
Podemos notar que las profesiones más ligadas a las mujeres son todas aquellas que se dedican a cuidar, mientras que en los hombres son palabras enfocadas al conocimiento

In [20]:
print(word_vectors.most_similar(positive=['man', 'brilliant'], negative=['woman']))
print()
print(word_vectors.most_similar(positive=['woman', 'brilliant'], negative=['man']))

[('superb', 0.770228922367096), ('fantastic', 0.6994621753692627), ('impressive', 0.6784907579421997), ('clever', 0.6677661538124084), ('spectacular', 0.6642494797706604), ('dazzling', 0.6544190049171448), ('terrific', 0.6470831036567688), ('amazing', 0.6459280848503113), ('magnificent', 0.6383018493652344), ('brilliantly', 0.6317154765129089)]

[('dazzling', 0.670451819896698), ('beautiful', 0.6414750218391418), ('gorgeous', 0.6371448040008545), ('superb', 0.6332093477249146), ('lovely', 0.61921226978302), ('witty', 0.6005319952964783), ('graceful', 0.5998649001121521), ('stunning', 0.5918648838996887), ('bright', 0.5887019634246826), ('elegant', 0.5836897492408752)]


### Analizando la palabra "Brilliant"
Notemos que con "brilliant" los atributos masculinos están todos relacionados al intelecto y a las ciencias, en cambio los atributos femeninos están todos relacionados con la apariencia

## Una forma de mitigar los sesgos de género al entrenar Modelos de lenguaje
Una forma de mitigar estos sesgos podría ser balancear la cantidad de atributos relacionados con los hombres y las mujeres. Por ejemplo, si la oración original menciona a "los ingenieros", el texto puede ser modificado o replicado para mencionar a "las ingenieras", logrando así que al final el corpus de entrenamiento tenga la misma cantidad de oraciones asociadas a un género u otro.